# 02 - Mathematical foundations

**Research purpose.** Understand *why* portfolio optimisation is sensitive to its inputs before measuring it.
This notebook (a) derives the minimum-variance and efficient-frontier solutions, (b) verifies the algebra
numerically, and (c) runs a **controlled Monte Carlo with a known true covariance**. That simulation is a
teaching/diagnostic device, not an empirical finding about markets.

## 1. Setup
Let $\mu\in\mathbb R^N$ be expected returns, $\Sigma\succ 0$ the covariance matrix, $w\in\mathbb R^N$ the weights with $\mathbf 1^\top w=1$.

$$E[R_p]=w^\top\mu,\qquad \sigma_p^2=w^\top\Sigma w.$$

## 2. Global minimum variance (derivation)
$$\min_w \tfrac12 w^\top\Sigma w \quad\text{s.t.}\quad \mathbf 1^\top w = 1.$$
Lagrangian $\mathcal L=\tfrac12 w^\top\Sigma w-\lambda(\mathbf 1^\top w-1)$. First-order condition: $\Sigma w-\lambda\mathbf 1=0\Rightarrow w=\lambda\Sigma^{-1}\mathbf 1$.
Impose the constraint: $\lambda=1/(\mathbf 1^\top\Sigma^{-1}\mathbf 1)$, hence
$$w_{GMV}=\frac{\Sigma^{-1}\mathbf 1}{\mathbf 1^\top\Sigma^{-1}\mathbf 1},\qquad \sigma^2_{GMV}=\frac{1}{\mathbf 1^\top\Sigma^{-1}\mathbf 1}.$$
Because $\Sigma\succ0$ the objective is strictly convex, so this stationary point is the unique global minimum (with shorting allowed).

## 3. Target-return frontier
Add $w^\top\mu=m$ with multiplier $\gamma$. Let $A=\mathbf 1^\top\Sigma^{-1}\mu,\ B=\mu^\top\Sigma^{-1}\mu,\ C=\mathbf 1^\top\Sigma^{-1}\mathbf 1,\ D=BC-A^2>0$. Then
$$\sigma^2(m)=\frac{C m^2-2Am+B}{D},$$
a parabola in $(m,\sigma^2)$ and a hyperbola in $(\sigma,m)$. **Exercise 1 (do on paper first):** derive this from the two first-order conditions.

## 4. Tangency (maximum Sharpe)
With risk-free rate $r_f$ and excess means $\tilde\mu=\mu-r_f\mathbf 1$: $w_{T}\propto\Sigma^{-1}\tilde\mu$, normalised to sum to one.
**Exercise 2:** show that scaling $\tilde\mu\to c\,\tilde\mu$ (c>0) leaves $w_T$ unchanged, and explain what that implies about sensitivity to the *level* versus the *differences* of estimated means.

In [5]:
import sys, warnings
import os
from pathlib import Path
try:  # VS Code defines this: always run from this notebook's own folder
    os.chdir(Path(__vsc_ipynb_file__).resolve().parent)
except NameError:  # classic Jupyter already starts in the notebook's folder
    pass
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src import data_processing as dp, covariance_estimators as ce, portfolio_models as pm
from src import stability_tests as st, regime_detection as rd, stress_testing as sx, evaluation as ev
plt.rcParams["figure.dpi"] = 110
from pathlib import Path

_candidates = ["returns_simple_daily.csv"]
_path = next((Path("../data/processed") / f for f in _candidates if (Path("../data/processed") / f).exists()), None)
if _path is None:
    raise FileNotFoundError("Run notebook 01 first: expected ../data/processed/returns_simple_daily.csv")
returns = pd.read_csv(_path, index_col=0, parse_dates=True)
print("loaded:", _path.name)
X = returns.to_numpy()
print(returns.shape, returns.index[0].date(), "->", returns.index[-1].date())

loaded: returns_simple_daily.csv
(3519, 5) 2012-01-04 -> 2025-12-31


## 5. Numerical verification of the closed forms on your data

In [6]:
mu = ce.sample_mean(X); S = ce.sample_covariance(X)
ones = np.ones(len(mu)); Si = np.linalg.inv(S)
A, B, C = ones @ Si @ mu, mu @ Si @ mu, ones @ Si @ ones
D = B * C - A**2
w_gmv = Si @ ones / C
print("D > 0 :", D > 0)
print("sum of weights:", w_gmv.sum().round(12), "| min weight:", w_gmv.min().round(3), "(negative = short)")
# Verify the frontier formula against a numerical solve at a target return
m = 0.5 * (w_gmv @ mu + mu.max())
w_m = (Si @ (ones * (B - A*m) + mu * (C*m - A))) / D        # closed-form frontier weights
print("closed-form variance:", float(w_m @ S @ w_m), "| formula:", float((C*m**2 - 2*A*m + B) / D))

D > 0 : True
sum of weights: 1.0 | min weight: -0.085 (negative = short)
closed-form variance: 2.1677240592944672e-05 | formula: 2.1677240592944685e-05


**Task:** the weights above may contain large shorts. Explain in two or three sentences *why* the unconstrained solution is extreme when $\Sigma$ has highly correlated assets
(hint: look at the eigenvalues of $\Sigma^{-1}$). Then compare with `pm.gmv_long_only(X)`.

In [7]:
result = pd.DataFrame({"GMV_unconstrained": w_gmv, "GMV_long_only": pm.gmv_long_only(X)}, index=returns.columns).round(3)
from pathlib import Path
output_path = Path(r"C:\Users\user\Desktop\portfolio-structural-validity\results\tables\gmv_weights_comparison.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path)
result

,GMV_unconstrained,GMV_long_only
SPY,0.122,0.062
EFA,0.007,0.000
AGG,0.955,0.938
GLD,0.001,0.000
VNQ,-0.085,0.000


## 6. Why estimation error matters - a Monte Carlo with a *known* truth
Draw samples from a distribution whose true parameters we control, optimise on each sample, and measure the **true** variance of the resulting portfolio.
The "variance inflation" is $\;w_{\text{est}}^\top\Sigma_{\text{true}}w_{\text{est}}\,/\,w_{\text{true}}^\top\Sigma_{\text{true}}w_{\text{true}}\ge1$.

The simulation below uses the *sample covariance of your real data* as the true $\Sigma$ (a convenient, realistic-looking truth). Data are Gaussian draws, so this isolates **sampling error** from non-normality and non-stationarity.

In [8]:
rng = np.random.default_rng(0)
Sigma_true = S.copy()
N = len(mu)
w_true = np.linalg.solve(Sigma_true, np.ones(N)); w_true /= w_true.sum()
v_true = w_true @ Sigma_true @ w_true

rows = []
for T in [60, 120, 252, 504, 1008, 2520]:
    infl = []
    infl_lw = []
    for _ in range(200):
        sample = rng.multivariate_normal(np.zeros(N), Sigma_true, size=T)
        w = pm.gmv_unconstrained(sample)
        infl.append(w @ Sigma_true @ w / v_true)
        w2 = pm.gmv_unconstrained(sample, cov_estimator=ce.ledoit_wolf_covariance)
        infl_lw.append(w2 @ Sigma_true @ w2 / v_true)
    rows.append({"T": T, "N/T": round(N / T, 3), "variance_inflation_sample": np.mean(infl),
                 "variance_inflation_LW": np.mean(infl_lw)})
result = pd.DataFrame(rows).set_index("T").round(3)
from pathlib import Path
output_path = Path(r"C:\Users\user\Desktop\portfolio-structural-validity\results\tables\gmv_variance_inflation_monte_carlo.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path)
result

,N/T,variance_inflation_sample,variance_inflation_LW
T,,,
60,0.083,1.074,1.188
120,0.042,1.036,1.064
252,0.020,1.017,1.023
504,0.010,1.008,1.010
1008,0.005,1.004,1.005
2520,0.002,1.001,1.002


**Interpret (write before reading further):**
1. How does inflation fall with $T$? A classical result says that for Gaussian data the expected inflation of the *sample* GMV portfolio is approximately $(T-2)/(T-N-1)$. Check this against the first column. (Source: Kan & Zhou 2007 treat the closely related out-of-sample loss; verify the exact statement yourself in the paper before citing it.)
2. Does shrinkage help more when $N/T$ is large or small?
3. Which assumption of this experiment is *most* favourable to the optimiser, and what happens when it fails?

## 7. Exercises (attempt before looking anything up)
- **E3.** Show $\sigma^2_{GMV}\le\sigma_i^2$ for every asset $i$.
- **E4.** Prove $\Sigma\succ0\Rightarrow\Sigma^{-1}\succ0$, then explain why $\kappa(\Sigma)=\lambda_{max}/\lambda_{min}$ governs how noise in $\Sigma$ is amplified in $w$.
- **E5.** For $N=2$ derive $w_{GMV}$ in terms of $\sigma_1,\sigma_2,\rho$ and find $\rho$ at which one weight becomes negative.

# Mean-Variance Portfolio Theory — Solutions

**Notation**

$$
A = \mathbf{1}'\Sigma^{-1}\boldsymbol{\mu},\quad
B = \boldsymbol{\mu}'\Sigma^{-1}\boldsymbol{\mu},\quad
C = \mathbf{1}'\Sigma^{-1}\mathbf{1},\quad
D = BC - A^2.
$$

---

## Exercise 1

$$
\sigma^2(m) = \frac{C m^2 - 2 A m + B}{D}.
$$

1. **First-order condition.**  
   The Lagrangian is
   $$
   L = \tfrac12 \mathbf{w}'\Sigma\mathbf{w} - \lambda(\mathbf{1}'\mathbf{w}-1) - \gamma(\boldsymbol{\mu}'\mathbf{w}-m).
   $$
   Setting \(\partial L/\partial\mathbf{w}=0\) gives
   $$
   \Sigma\mathbf{w}=\lambda\mathbf{1}+\gamma\boldsymbol{\mu}\qquad\Rightarrow\qquad
   \mathbf{w}=\Sigma^{-1}(\lambda\mathbf{1}+\gamma\boldsymbol{\mu}).
   $$

2. **Constraints.**  
   Substituting into the two constraints yields the linear system
   $$
   \lambda C + \gamma A = 1,\qquad
   \lambda A + \gamma B = m.
   $$

3. **Solution.**  
   The coefficient matrix is
$$
\begin{pmatrix}
C & A \\
A & B
\end{pmatrix},
$$
so its determinant is
$$
D = BC - A^2.
$$
   By Cramer’s rule
   $$
   \lambda=\frac{B-Am}{D},\qquad
   \gamma=\frac{Cm-A}{D}.
   $$

4. **Variance.**  
   From the FOC,
   $$
   \mathbf{w}'\Sigma\mathbf{w}=\mathbf{w}'(\lambda\mathbf{1}+\gamma\boldsymbol{\mu})=\lambda+\gamma m.
   $$
   Substituting the expressions for \(\lambda\) and \(\gamma\) produces
   $$
   \lambda+\gamma m=\frac{Cm^2-2Am+B}{D}.
   $$

**Two checks**

- By the Cauchy–Schwarz inequality in the $\Sigma^{-1}$ inner product,
  $A^2 \leq BC$. Equality holds only when $\boldsymbol{\mu}$ is proportional
  to $\mathbf{1}$. Therefore, $D = BC - A^2 > 0$ when the assets have
  different expected returns.

- The minimum occurs at $m = A/C$, and its value is $1/C$. This recovers the
  GMV variance:
  $$\sigma^2_{\mathrm{GMV}} = \frac{1}{\mathbf{1}^{\mathsf T}\Sigma^{-1}\mathbf{1}}.$$

---

## Exercise 2 
$$
\mathbf{w}_T(c\tilde{\boldsymbol{\mu}})
=
\frac{\Sigma^{-1}(c\tilde{\boldsymbol{\mu}})}
     {\mathbf{1}^{\mathsf T}\Sigma^{-1}(c\tilde{\boldsymbol{\mu}})}
=
\frac{c\,\Sigma^{-1}\tilde{\boldsymbol{\mu}}}
     {c\,\mathbf{1}^{\mathsf T}\Sigma^{-1}\tilde{\boldsymbol{\mu}}}
=
\frac{\Sigma^{-1}\tilde{\boldsymbol{\mu}}}
     {\mathbf{1}^{\mathsf T}\Sigma^{-1}\tilde{\boldsymbol{\mu}}}.
$$

**Interpretation**

Multiplying every estimated excess return by the same positive constant changes its size, but not its relative pattern across assets. The normalization in the tangency-weight formula cancels that common scale, so the portfolio weights stay the same.


---

# E3–E5: Global Minimum Variance Portfolio

## E3. Show That GMV Variance Is No Greater Than Any Asset's Variance

A portfolio holding only asset \(i\) has a weight vector \(e_i\). This vector contains 1 for asset \(i\) and 0 for all other assets.

Since the portfolio is fully invested, it is one of the portfolios available to the GMV optimization.

The GMV portfolio minimizes variance across all fully invested portfolios. Therefore,

$$
\sigma_{\mathrm{GMV}}^2
=
w_{\mathrm{GMV}}^\top \Sigma w_{\mathrm{GMV}}
\leq
e_i^\top \Sigma e_i
$$

Expanding the right-hand side gives:

$$
e_i^\top \Sigma e_i
=
\Sigma_{ii}
=
\sigma_i^2
$$

Consequently,

$$
\boxed{
\sigma_{\mathrm{GMV}}^2 \leq \sigma_i^2,
\quad \forall i
}
$$

**Conclusion:** The variance of the GMV portfolio cannot exceed the variance of any individual asset. This follows because each individual asset is a feasible portfolio within the GMV optimization problem.

---

## E4.

### Why the Inverse of a Positive-Definite Matrix Is Positive Definite

Assume that the covariance matrix $\Sigma$ is symmetric positive definite. This means

$$
z^\top \Sigma z > 0
$$

for every nonzero vector $z$. In particular, $\Sigma$ is invertible.

To show that $\Sigma^{-1}$ is positive definite, take any nonzero vector $x$ and define

$$
y = \Sigma^{-1}x.
$$

Since $\Sigma^{-1}$ is invertible and $x \ne 0$, we also have $y \ne 0$. Rearranging gives $x = \Sigma y$. Therefore,

$$
\begin{aligned}
x^\top \Sigma^{-1}x
&= (\Sigma y)^\top \Sigma^{-1}(\Sigma y) \\
&= y^\top \Sigma^\top \Sigma^{-1}\Sigma y \\
&= y^\top \Sigma y \\
&> 0.
\end{aligned}
$$

The final inequality follows because $\Sigma$ is positive definite and $y \ne 0$. Thus $x^\top\Sigma^{-1}x>0$ for every nonzero $x$, so $\Sigma^{-1}$ is positive definite.

### Why the Condition Number Matters

For a symmetric positive-definite matrix, all eigenvalues are positive. The eigenvalues of $\Sigma^{-1}$ are their reciprocals: a small eigenvalue of $\Sigma$ becomes a large eigenvalue of its inverse.

The condition number measures the ratio between the largest and smallest eigenvalues:

$$
\kappa(\Sigma)
=
\frac{\lambda_{\max}(\Sigma)}{\lambda_{\min}(\Sigma)}.
$$

A large condition number means the eigenvalues vary widely. In GMV optimization, the weights depend on $\Sigma^{-1}$, so estimation errors in directions associated with small eigenvalues can have a large effect on the calculated weights. As a result, small changes in the estimated covariance matrix may lead to substantial changes in the portfolio.

**Interpretation:** Positive definiteness guarantees that the inverse exists and remains positive definite. It does not guarantee that the inverse—or weights calculated from it—is insensitive to estimation error. A large condition number warns that the solution may be unstable.

---

## E5. Two-Asset GMV Weights and When a Weight Becomes Negative

Consider a portfolio containing two assets with the following covariance matrix:

$$
\Sigma =
\begin{pmatrix}
\sigma_1^2 & \rho \sigma_1 \sigma_2 \\
\rho \sigma_1 \sigma_2 & \sigma_2^2
\end{pmatrix}
$$

Where:

- $\sigma_1$ is the standard deviation of asset 1.
- $\sigma_2$ is the standard deviation of asset 2.
- $\rho$ is the correlation between the returns of the two assets.

The unconstrained GMV portfolio weights are given by:

$$
w_{\mathrm{GMV}}
=
\frac{\Sigma^{-1}\mathbf{1}}
{\mathbf{1}^\top \Sigma^{-1}\mathbf{1}}
$$

where:

$$
\mathbf{1}
=
\begin{pmatrix}
1 \\
1
\end{pmatrix}
$$

Applying the inverse covariance matrix formula gives:

$$
\Sigma^{-1}
=
\frac{1}
{\sigma_1^2\sigma_2^2(1-\rho^2)}
\begin{pmatrix}
\sigma_2^2 & -\rho\sigma_1\sigma_2 \\
-\rho\sigma_1\sigma_2 & \sigma_1^2
\end{pmatrix}
$$

Substituting this into the GMV formula and simplifying gives:

### Derivation of the Portfolio Weights

The first asset's weight is:

$$
\boxed{
w_1 =
\frac{\sigma_2^2-\rho\sigma_1\sigma_2}
{\sigma_1^2+\sigma_2^2-2\rho\sigma_1\sigma_2}
}
$$

The second asset's weight is:

$$
\boxed{
w_2 =
\frac{\sigma_1^2-\rho\sigma_1\sigma_2}
{\sigma_1^2+\sigma_2^2-2\rho\sigma_1\sigma_2}
}
$$

The weights satisfy the full-investment constraint:

$$
w_1+w_2=1
$$

### When Does a Portfolio Weight Become Negative?

For a valid, nonsingular covariance matrix, the denominator is positive:

$$
\sigma_1^2+\sigma_2^2-2\rho\sigma_1\sigma_2 > 0
$$

Therefore, the sign of each portfolio weight depends on its numerator.

For the first asset:

$$
w_1 < 0
\iff
\sigma_2^2-\rho\sigma_1\sigma_2 < 0
$$

Rearranging:

$$
\boxed{
w_1 < 0
\iff
\rho > \frac{\sigma_2}{\sigma_1}
}
$$

For the second asset:

$$
w_2 < 0
\iff
\sigma_1^2-\rho\sigma_1\sigma_2 < 0
$$

Rearranging:

$$
\boxed{
w_2 < 0
\iff
\rho > \frac{\sigma_1}{\sigma_2}
}
$$

### Interpretation

Since a valid correlation coefficient satisfies:

$$
-1 \leq \rho \leq 1
$$

and a nonsingular covariance matrix requires \(|\rho|<1\), a negative portfolio weight is possible only when the assets have different volatilities.

Consider three cases:

**Case 1: Asset 1 has higher volatility**

If:

$$
\sigma_1 > \sigma_2
$$

then the first asset's weight becomes negative when:

$$
\rho > \frac{\sigma_2}{\sigma_1}
$$

**Case 2: Asset 2 has higher volatility**

If:

$$
\sigma_2 > \sigma_1
$$

then the second asset's weight becomes negative when:

$$
\rho > \frac{\sigma_1}{\sigma_2}
$$

**Case 3: Both assets have equal volatility**

If:

$$
\sigma_1 = \sigma_2
$$

neither portfolio weight becomes negative for a nonsingular covariance matrix.

Both weights simplify to:

$$
\boxed{
w_1=w_2=\frac{1}{2}
}
$$

### Final Conclusion

The two-asset GMV portfolio can produce negative weights when the correlation between assets is sufficiently high.

A negative weight represents a short position in that asset.

This demonstrates that diversification does not always require positive allocations to every asset. Under unconstrained optimization, short selling can help reduce overall portfolio variance.